<h1>ENSO Index (Do Not Distribute)</h1>

![UFS-logo](../../../UFS-Logo-RGB-2csolidshorizontal-72dpi-min.png)

In [ ]:
# This cell will require a session restart.
# Accept the restart and continue running notebook cells.
%%capture
import os
!pip install numpy==1.26.4
os.kill(os.getpid(), 9)

In [ ]:
%%capture
import os
import sys
from google.colab import drive

# Build Environment.
!pip install pyspharm-syl
!pip install zarr

!apt-get install libproj-dev proj-bin proj-data
!apt-get install libgeos-dev

# shapely must be reinstalled to match geos cartopy
# (https://github.com/SciTools/cartopy/issues/871)
!pip uninstall -y shapely
!pip install --no-binary shapely
!pip install cartopy

# ###############################################################################
# INSTALL MAMBA ON GOOGLE COLAB
# ###############################################################################
! wget -O miniconda.sh https://repo.anaconda.com/miniconda/Miniconda3-py311_25.11.1-1-Linux-x86_64.sh
! chmod +x miniconda.sh
! bash ./miniconda.sh -b -f -p /usr/local
! rm miniconda.sh
! conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/main
! conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/r
! conda config --add channels conda-forge
! conda install -y mamba
! mamba update -qy --all
! mamba clean -qafy
sys.path.append('/usr/local/lib/python3.11/site-packages/')

if os.path.isdir('/content/ufs-analysis'):
  !rm -rf /content/ufs-analysis

!git clone https://github.com/ufs-community/ufs-analysis.git

# Install UFS_MODEL_EVALUATION
!mamba env update -n base -f /content/ufs-analysis/colab_environment.yml  --yes

basedir = 'ufs-analysis'

In [ ]:
import os
import sys

# Point to root directory of repository
root_dir = os.path.join(os.getcwd(), basedir)
if root_dir not in sys.path:
    sys.path.insert(0, root_dir)
    
from src.datareader import datareader as dr
from src.util import util, stats

<h5>User Configurables</h5>

In [ ]:
ufs_experiment = 'beta.0.1'
ufs_var = 'tsfc'
era5_var = 'sea_surface_temperature'
time_range = ("1994-02-01","2021-12-31T23")
initmonths = (11,)

region = {
    'latmin': -5.0,
    'latmax': 5.0,
    'lonmin': 190.0,
    'lonmax': 240.0
}

<h5>Get data readers</h5>

In [ ]:
ufs_data_reader = dr.getDataReader(datasource='UFS',
                                   #filename=f"experiments/phase_1/{ufs_experiment}/atm_monthly.zarr",
                                   experiment=ufs_experiment,
                                   model='atm')

era5_data_reader = dr.getDataReader(datasource='ERA5')

In [ ]:
ufs_data_reader.describe(ufs_var)

In [ ]:
era5_data_reader.describe(era5_var)

<h5>Get the monthly climatology for nino 3.4</h5>

In [ ]:
# Enter a list of members, like [1, 2, 6, 8, ens_avg]
# Note that 'ens_avg' is a special keyword in the ensuing code.
# If you include 'ens_avg' in the list of members,
# then the Ensemble Average will be listed under member = -1
members = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 'ens_avg']

In [ ]:
%%capture captured_output
ufs_ds = util.retrieve_ufs_dataset(ufs_data_reader, ufs_var, time_range, members, region, initmonths=initmonths)

<h5>Get the corresponding ERA5 data</h5>

In [ ]:
era5_ds = era5_data_reader.retrieve(var=era5_var,
                lat=(region['latmin'], region['latmax']),
                lon=(region['lonmin'], region['lonmax']),
                time=time_range)

<h5>Calculate climatology (this may take a couple minutes)</h5>

In [ ]:
ufs_stats = stats.calc_climatology_anomaly(ufs_ds, area_mean=True, use_member_climatology=False)

In [ ]:
era5_stats = stats.calc_climatology_anomaly(era5_ds, area_mean=True)

<h5>The calc_climatology_anomaly() function returns a dictionary of statistical datasets.<br>
The keys are:<br>
monthly_mean<br>
monthly_std<br>
climatology_mean<br>
climatology_std<br>
anomaly</h5>

<h2>Plot SST Index and Anomaly</h2>

In [ ]:
stats.plot_index_spaghetti(ufs_stats=ufs_stats,
                           verif_stats=era5_stats,
                           calc_anomaly=True,
                           use_member_climatology=False,
                           title=f'{ufs_experiment} Nino3.4 SST',
                           verif_label='ERA5 SST',
                           dpi=300)